# KG1 v51 PERFECT — 9500 Solver-Enhanced CoTs (100% ACC)

**Base:** v30 config (scored 0.68) + 9500 solver-verified examples

**Mudancas vs v30:**
- 9500 exemplos com CoTs (vs 5000 sem CoT)
- Solver deterministico: bit 79%, gravity/unit/numeral 100%
- Answer-assisted CoTs para cipher + equation
- Smart-strip no submit (mantém shared_experts = +0.06)

**Config IDENTICA ao v30:** r=32, alpha=16, lr=5e-5, grad_accum=8

**Score esperado: 0.74-0.80**

In [1]:
#@title Cell 1 — Install Dependencies
!pip install -q peft datasets accelerate trl huggingface_hub safetensors pandas
!pip install -q causal-conv1d mamba-ssm 2>/dev/null || echo 'mamba-ssm install failed (ok for Blackwell)'

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 65.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 5.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
mamba-ssm install failed (ok for Blackwell)
PyTorch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


In [2]:
#@title Cell 2 — Auth + Config
import subprocess, sys, os, json, random, time, zipfile, shutil, re
from datetime import datetime, timezone
from collections import Counter

# Monkey-patch
try:
    from transformers.utils.import_utils import is_flash_attn_greater_or_equal_2_10
except ImportError:
    import transformers.utils.import_utils as _tiu
    _tiu.is_flash_attn_greater_or_equal_2_10 = lambda: False
    print('Patched: is_flash_attn_greater_or_equal_2_10')

import torch
import pandas as pd
from huggingface_hub import HfApi, login, hf_hub_download

# Auth
def _get_secret(*names):
    try:
        from google.colab import userdata
        for n in names:
            v = userdata.get(n)
            if v: return v
    except Exception: pass
    for n in names:
        v = os.environ.get(n)
        if v: return v
    return ''

HF_TOKEN = _get_secret('HF_KEY', 'HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF login OK')

KAGGLE_USERNAME = _get_secret('KAGGLE_USERNAME') or 'felipe1983'
KAGGLE_KEY = _get_secret('KAGGLE_KEY')
if KAGGLE_KEY:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    kpath = os.path.expanduser('~/.kaggle/kaggle.json')
    with open(kpath, 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.chmod(kpath, 0o600)
    print(f'Kaggle: {KAGGLE_USERNAME}')

api = HfApi(token=HF_TOKEN) if HF_TOKEN else HfApi()

# Config
DATA_REPO = 'felipesp1983/kg1-nemotron-training'
OUTPUT_REPO = 'felipesp1983/kg1-nemotron-lora-v51-perfect'
MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
COMPETITION = 'nvidia-nemotron-model-reasoning-challenge'

N_EXAMPLES = 5000  #@param {type:"integer"}
N_EPOCHS = 2  #@param {type:"integer"}
SUBMIT_STEPS = [200, 400, 600, 800, 1000, 1200]

CONFIG = {
    'lora_rank': 32,
    'lora_alpha': 16,
    'lora_dropout': 0.05,
    'target_modules': 'all-linear',
    'learning_rate': 5e-5,
    'per_device_batch_size': 1,
    'gradient_accumulation_steps': 8,
    'max_length': 1024,
    'warmup_ratio': 0.05,
    'weight_decay': 0.01,
    'lr_scheduler': 'cosine',
    'optim': 'adamw_torch',
    'output_dir': '/tmp/kg1_output/v51',
}

# Triton patch
try:
    for ptxas_src in ['/usr/local/cuda-12.8/bin/ptxas', '/usr/local/cuda/bin/ptxas']:
        if os.path.exists(ptxas_src):
            target = os.path.join(os.path.dirname(shutil.which('python') or '/usr/bin/python'), 'ptxas')
            if not os.path.exists(target):
                shutil.copy2(ptxas_src, target)
                print(f'Triton ptxas patched: {target}')
            break
except Exception: pass

print(f'Config: {N_EXAMPLES} examples, {N_EPOCHS} epochs, LR={CONFIG["learning_rate"]}')
print(f'LoRA: r={CONFIG["lora_rank"]}, alpha={CONFIG["lora_alpha"]}')

HF login OK
Kaggle: felipe1983
Triton ptxas patched: /usr/local/bin/ptxas
Config: 5000 examples, 2 epochs, LR=5e-05
LoRA: r=32, alpha=16


In [3]:
#@title Cell 3 — Load Data (9500 solver-enhanced CoTs)
print('=== Loading v51 PERFECT data ===')

# Try HF download first, then GitHub clone fallback
data_loaded = False
all_examples = []

# Method 1: Download from HF
try:
    hf_hub_download(repo_id=DATA_REPO, repo_type='dataset',
                    filename='data/sft_v51_perfect.jsonl', local_dir='/tmp/kg1_data')
    with open('/tmp/kg1_data/data/sft_v51_perfect.jsonl') as f:
        for line in f:
            all_examples.append(json.loads(line))
    data_loaded = True
    print(f'Loaded from HF: {len(all_examples)} examples')
except Exception as e:
    print(f'HF download failed: {e}')

# Method 2: GitHub clone
if not data_loaded:
    try:
        if not os.path.exists('/tmp/kg1_repo'):
            !git clone https://github.com/davisdenner/KG1-NVIDIA.git /tmp/kg1_repo 2>/dev/null
        jsonl_path = '/tmp/kg1_repo/data/sft_v51_perfect.jsonl'
        if os.path.exists(jsonl_path):
            with open(jsonl_path) as f:
                for line in f:
                    all_examples.append(json.loads(line))
            data_loaded = True
            print(f'Loaded from GitHub: {len(all_examples)} examples')
    except Exception as e:
        print(f'GitHub clone failed: {e}')

# Method 3: Upload manually
if not data_loaded:
    print('\n!!! DATA NOT FOUND !!!')
    print('Please upload sft_v51_perfect.jsonl to Colab:')
    print('  1. Click folder icon (left sidebar)')
    print('  2. Upload sft_v51_perfect.jsonl')
    print('  3. Re-run this cell')
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded:
            with open(fname) as f:
                for line in f:
                    all_examples.append(json.loads(line))
            data_loaded = True
            print(f'Loaded from upload: {len(all_examples)} examples')
    except Exception:
        pass

if not data_loaded:
    # Final fallback: download train.csv and use raw answers (v30 style)
    print('Falling back to train.csv (v30 style)...')
    hf_hub_download(repo_id=DATA_REPO, repo_type='dataset',
                    filename='data/train.csv', local_dir='/tmp/kg1_data')
    train_df = pd.read_csv('/tmp/kg1_data/data/train.csv')
    for _, row in train_df.iterrows():
        all_examples.append({
            'prompt': row['prompt'] + '\nPut your final answer inside \\boxed{}.',
            'completion': f'\\boxed{{{row["answer"]}}}',
            'family': 'unknown',
        })
    print(f'Loaded {len(all_examples)} raw examples (v30 fallback)')

# Classify families
def classify(text):
    p = text.lower()
    if 'bit manipulation' in p: return 'bit'
    if 'gravitational' in p: return 'grav'
    if 'unit conversion' in p or 'measurement' in p: return 'unit'
    if 'numeral' in p: return 'num'
    if 'encryption' in p: return 'enc'
    if 'transformation' in p: return 'eq'
    return 'other'

# Stratified sampling
print(f'\n=== Sampling {N_EXAMPLES} examples (type-weighted) ===')
random.seed(42)

by_family = {}
for ex in all_examples:
    fam = ex.get('family') or classify(ex.get('prompt', ''))
    by_family.setdefault(fam, []).append(ex)

# Type-weighted: oversample hard families
shares = {'grav': 1.0, 'unit': 1.0, 'num': 1.0, 'enc': 1.0,
          'cipher': 1.0, 'bit': 1.5, 'eq': 2.5, 'equation': 2.5,
          'gravity': 1.0, 'numeral': 1.0}
total_shares = sum(shares.get(f, 1.0) for f in by_family)
base_n = N_EXAMPLES / total_shares

examples = []
for fam, pool in by_family.items():
    mult = shares.get(fam, 1.0)
    n_want = int(base_n * mult)
    if n_want <= len(pool):
        selected = random.sample(pool, n_want)
    else:
        selected = pool + random.choices(pool, k=n_want - len(pool))
    examples.extend(selected)

random.shuffle(examples)
examples = examples[:N_EXAMPLES]

# Format for SFTTrainer
formatted = []
for ex in examples:
    formatted.append({
        'messages': [
            {'role': 'user', 'content': ex.get('prompt', '')},
            {'role': 'assistant', 'content': ex.get('completion', '')},
        ]
    })

fam_counts = Counter(classify(e['messages'][0]['content']) for e in formatted)
print(f'Final dataset: {len(formatted)} examples')
for fam, cnt in sorted(fam_counts.items()):
    print(f'  {fam}: {cnt} ({cnt/len(formatted)*100:.1f}%)')

=== Loading v51 PERFECT data ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


sft_v51_perfect.jsonl: 0.00B [00:00, ?B/s]

Loaded from HF: 9500 examples

=== Sampling 5000 examples (type-weighted) ===
Final dataset: 4999 examples
  bit: 937 (18.7%)
  enc: 625 (12.5%)
  eq: 1562 (31.2%)
  grav: 625 (12.5%)
  num: 625 (12.5%)
  unit: 625 (12.5%)


In [7]:
# Cell 3.5 — Fix mamba-ssm for Blackwell
import sys, types

# Create stub mamba_ssm module
mamba_ssm = types.ModuleType('mamba_ssm')
mamba_ssm.__version__ = '0.0.0'

ops = types.ModuleType('mamba_ssm.ops')
triton_mod = types.ModuleType('mamba_ssm.ops.triton')
ln_gated = types.ModuleType('mamba_ssm.ops.triton.layernorm_gated')
ssd = types.ModuleType('mamba_ssm.ops.triton.selective_state_update')
ssd_ref = types.ModuleType('mamba_ssm.ops.triton.ssd_combined')
utils = types.ModuleType('mamba_ssm.utils')
gen = types.ModuleType('mamba_ssm.utils.generation')

class _Stub:
    def __init__(self, *a, **kw): pass
    def __call__(self, *a, **kw): return a[0] if a else None

for mod in [ln_gated, ssd, ssd_ref, gen]:
    for attr in ['RMSNormGated', 'rmsnorm_fn', 'selective_state_update',
                 'mamba_chunk_scan_combined', 'mamba_split_conv1d_scan_combined',
                 'InferenceParams', 'GenerationMixin']:
        setattr(mod, attr, _Stub)

triton_mod.layernorm_gated = ln_gated
triton_mod.selective_state_update = ssd
triton_mod.ssd_combined = ssd_ref
ops.triton = triton_mod
mamba_ssm.ops = ops
mamba_ssm.utils = utils
utils.generation = gen

sys.modules['mamba_ssm'] = mamba_ssm
sys.modules['mamba_ssm.ops'] = ops
sys.modules['mamba_ssm.ops.triton'] = triton_mod
sys.modules['mamba_ssm.ops.triton.layernorm_gated'] = ln_gated
sys.modules['mamba_ssm.ops.triton.selective_state_update'] = ssd
sys.modules['mamba_ssm.ops.triton.ssd_combined'] = ssd_ref
sys.modules['mamba_ssm.utils'] = utils
sys.modules['mamba_ssm.utils.generation'] = gen

print("mamba-ssm STUB injected for Blackwell")


mamba-ssm STUB injected for Blackwell


In [9]:
# Fix causal_conv1d for Blackwell
import sys, types

class _Stub:
    def __init__(self, *a, **kw): pass
    def __call__(self, *a, **kw): return a[0] if a else None
    def __getattr__(self, name): return _Stub()

cc1d = types.ModuleType('causal_conv1d')
cc1d.causal_conv1d_fn = _Stub()
cc1d.causal_conv1d_update = _Stub()
cc1d.__spec__ = types.ModuleType('_spec')
cc1d.__spec__.name = 'causal_conv1d'
cc1d.__spec__.origin = 'stub'

sys.modules['causal_conv1d'] = cc1d
sys.modules['causal_conv1d.causal_conv1d_interface'] = cc1d

print("causal_conv1d STUB injected")


causal_conv1d STUB injected


In [ ]:
#@title Cell 4 — Load Model + LoRA (Blackwell Fix)

# ============================================================
# FIX BLACKWELL: Inject mamba-ssm stub BEFORE model load
# ============================================================
import sys, types

if 'mamba_ssm' not in sys.modules:
    mamba_ssm = types.ModuleType('mamba_ssm')
    mamba_ssm.__version__ = '0.0.0'

    ops = types.ModuleType('mamba_ssm.ops')
    triton_mod = types.ModuleType('mamba_ssm.ops.triton')
    ln_gated = types.ModuleType('mamba_ssm.ops.triton.layernorm_gated')
    ssd = types.ModuleType('mamba_ssm.ops.triton.selective_state_update')
    ssd_ref = types.ModuleType('mamba_ssm.ops.triton.ssd_combined')
    utils_mod = types.ModuleType('mamba_ssm.utils')
    gen_mod = types.ModuleType('mamba_ssm.utils.generation')

    class _Stub:
        def __init__(self, *a, **kw): pass
        def __call__(self, *a, **kw): return a[0] if a else None
        def __getattr__(self, name): return _Stub()

    for mod in [ln_gated, ssd, ssd_ref, gen_mod]:
        for attr in ['RMSNormGated', 'rmsnorm_fn', 'selective_state_update',
                     'mamba_chunk_scan_combined', 'mamba_split_conv1d_scan_combined',
                     'InferenceParams', 'GenerationMixin']:
            setattr(mod, attr, _Stub)

    triton_mod.layernorm_gated = ln_gated
    triton_mod.selective_state_update = ssd
    triton_mod.ssd_combined = ssd_ref
    ops.triton = triton_mod
    mamba_ssm.ops = ops
    mamba_ssm.utils = utils_mod
    utils_mod.generation = gen_mod

    sys.modules['mamba_ssm'] = mamba_ssm
    sys.modules['mamba_ssm.ops'] = ops
    sys.modules['mamba_ssm.ops.triton'] = triton_mod
    sys.modules['mamba_ssm.ops.triton.layernorm_gated'] = ln_gated
    sys.modules['mamba_ssm.ops.triton.selective_state_update'] = ssd
    sys.modules['mamba_ssm.ops.triton.ssd_combined'] = ssd_ref
    sys.modules['mamba_ssm.utils'] = utils_mod
    sys.modules['mamba_ssm.utils.generation'] = gen_mod
    print("mamba-ssm STUB injected for Blackwell")

# Also stub causal_conv1d if missing
if 'causal_conv1d' not in sys.modules:
    cc1d = types.ModuleType('causal_conv1d')
    cc1d.causal_conv1d_fn = _Stub()
    cc1d.causal_conv1d_update = _Stub()
    sys.modules['causal_conv1d'] = cc1d
    print("causal_conv1d STUB injected")

# ============================================================
# LOAD MODEL
# ============================================================
print('=== Loading model (BF16) ===')
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

_gpu_cap = float(f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}')
IS_BLACKWELL = _gpu_cap >= 10.0

_model_kwargs = dict(
    device_map={'': 0},
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

if IS_BLACKWELL:
    _cfg = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if hasattr(_cfg, 'use_mamba_kernels'):
        _cfg.use_mamba_kernels = False
        _model_kwargs['config'] = _cfg
        print(f'[BLACKWELL sm_{int(_gpu_cap*10)}] Mamba CUDA kernels DISABLED (PyTorch native path)')
        print(f'Training ~2-3x slower but CORRECT on Blackwell')

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **_model_kwargs)

fp_count = 0
for module in model.modules():
    if hasattr(module, 'is_fast_path_available'):
        module.is_fast_path_available = False
        fp_count += 1
print(f'Model: {model.num_parameters()/1e9:.1f}B params, fast path disabled ({fp_count})')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizer: vocab={tokenizer.vocab_size}')

# ============================================================
# APPLY LORA
# ============================================================
print(f'\n=== Applying LoRA (r={CONFIG["lora_rank"]}, alpha={CONFIG["lora_alpha"]}) ===')
from peft import LoraConfig, get_peft_model

model.enable_input_require_grads()
lora_config = LoraConfig(
    r=CONFIG['lora_rank'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=CONFIG['target_modules'],
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


=== Loading model (BF16) ===
[BLACKWELL sm_120] Mamba CUDA kernels DISABLED (PyTorch native path)
Training ~2-3x slower but CORRECT on Blackwell


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

In [ ]:
#@title Cell 5 — Prepare Dataset
print('=== Preparing tokenized dataset ===')
from datasets import Dataset

texts = []
for ex in formatted:
    text = tokenizer.apply_chat_template(
        ex['messages'], tokenize=False, add_generation_prompt=False,
    )
    texts.append(text)

ds = Dataset.from_dict({'text': texts})
print(f'Dataset: {len(ds)} examples')

sample_lens = [len(tokenizer(t)['input_ids']) for t in texts[:100]]
print(f'Token lengths (100 samples): min={min(sample_lens)}, max={max(sample_lens)}, mean={sum(sample_lens)/len(sample_lens):.0f}')

In [ ]:
#@title Cell 6 — Train
import math
print('=== Training v51 PERFECT ===')
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

os.makedirs(CONFIG['output_dir'], exist_ok=True)

training_args = SFTConfig(
    output_dir=CONFIG['output_dir'],
    dataset_text_field='text',
    max_length=CONFIG['max_length'],
    packing=False,
    num_train_epochs=N_EPOCHS,
    per_device_train_batch_size=CONFIG['per_device_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    lr_scheduler_type=CONFIG['lr_scheduler'],
    optim=CONFIG['optim'],
    bf16=True,
    logging_steps=5,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=15,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to='none',
    dataloader_num_workers=0,
    max_grad_norm=1.0,
)

class AutoSubmitCallback(TrainerCallback):
    def __init__(self, repo_id, submit_steps):
        self.repo_id = repo_id
        self.submit_steps = set(submit_steps)
        self.submitted = set()
        self.api = HfApi(token=HF_TOKEN) if HF_TOKEN else HfApi()
        try: self.api.create_repo(repo_id, private=True, exist_ok=True)
        except: pass

    def on_save(self, args, state, control, **kwargs):
        import glob as g
        step = state.global_step
        loss_val = 'N/A'
        if state.log_history:
            for entry in reversed(state.log_history):
                if 'loss' in entry:
                    loss_val = entry['loss']
                    break
        ckpts = sorted(g.glob(f'{args.output_dir}/checkpoint-*'))
        if not ckpts: return
        ckpt_dir = ckpts[-1]
        try:
            self.api.upload_folder(
                folder_path=ckpt_dir, path_in_repo=f'checkpoint-{step}',
                repo_id=self.repo_id,
                commit_message=f'Step {step} | Loss {loss_val} | Epoch {state.epoch:.2f}',
            )
            print(f'\n>>> HF upload OK: step {step}, loss={loss_val}')
        except Exception as e:
            print(f'\n>>> HF upload FAILED: {e}')
        if step in self.submit_steps and step not in self.submitted:
            print(f'  [INFO] Step {step} uploaded. Submit manually with smart_strip_submit.py')
            self.submitted.add(step)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        loss = logs.get('loss', 0)
        if isinstance(loss, float) and (math.isnan(loss) or math.isinf(loss)):
            print(f'\n!!! CRITICAL: Loss NaN/Inf at step {state.global_step}')
            control.should_training_stop = True
            return
        if isinstance(loss, (int, float)) and loss > 30.0 and state.global_step > 5:
            print(f'\n!!! CRITICAL: Loss explosion {loss:.2f} at step {state.global_step}')
            control.should_training_stop = True
            return
        if state.global_step == 10:
            if loss > 8.0:
                print(f'\n!!! ALERT: Loss at step 10 = {loss:.2f} (very high!)')
            elif loss > 5.0:
                print(f'\n! WARNING: Loss at step 10 = {loss:.2f} (elevated)')
            else:
                print(f'\n>>> Loss at step 10 = {loss:.2f} (good)')

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    processing_class=tokenizer,
    args=training_args,
    callbacks=[AutoSubmitCallback(OUTPUT_REPO, SUBMIT_STEPS)],
)

total_steps = (len(ds) // CONFIG['gradient_accumulation_steps']) * N_EPOCHS
est_h = total_steps * 55 / 3600
print(f'Steps: ~{total_steps}, Est time: ~{est_h:.1f}h')
print(f'Submit steps: {sorted(SUBMIT_STEPS)}')

start = time.time()
try:
    trainer.train()
except Exception as e:
    print(f'\n!!! Training error: {e}')
    try:
        model.save_pretrained(CONFIG['output_dir'])
        tokenizer.save_pretrained(CONFIG['output_dir'])
        api.upload_folder(folder_path=CONFIG['output_dir'], repo_id=OUTPUT_REPO,
                         path_in_repo='emergency', commit_message=f'Emergency: {str(e)[:80]}')
    except: pass

elapsed = time.time() - start
print(f'\nTraining complete: {elapsed/3600:.2f}h')

In [ ]:
#@title Cell 7 — Save Final + Upload
print('=== Saving final adapter ===')
model.save_pretrained(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])

final_loss = 'N/A'
if trainer.state.log_history:
    for entry in reversed(trainer.state.log_history):
        if 'loss' in entry:
            final_loss = entry['loss']
            break

status = {
    'version': 'v51_perfect',
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'examples': len(formatted),
    'epochs': N_EPOCHS,
    'lr': CONFIG['learning_rate'],
    'lora_rank': CONFIG['lora_rank'],
    'lora_alpha': CONFIG['lora_alpha'],
    'training_time_h': elapsed / 3600,
    'final_loss': final_loss,
    'total_steps': trainer.state.global_step,
}
with open(f'{CONFIG["output_dir"]}/adapter_status.json', 'w') as f:
    json.dump(status, f, indent=2)

print(f'\n=== Uploading final to HF ===')
try:
    api.create_repo(OUTPUT_REPO, private=True, exist_ok=True)
    api.upload_folder(
        folder_path=CONFIG['output_dir'], repo_id=OUTPUT_REPO,
        commit_message=f'v51 FINAL: {len(formatted)}ex, {N_EPOCHS}ep, loss={final_loss}',
    )
    print(f'Uploaded: https://huggingface.co/{OUTPUT_REPO}')
except Exception as e:
    print(f'Upload failed: {e}')

print(f'\n{"="*60}')
print(f'  v51 PERFECT TRAINING COMPLETE')
print(f'  Examples: {len(formatted)} | Epochs: {N_EPOCHS}')
print(f'  Final loss: {final_loss}')
print(f'  Steps: {trainer.state.global_step}')
print(f'  Time: {elapsed/3600:.2f}h')
print(f'  NEXT: Submit step 400 with smart-strip')
print(f'{"="*60}')

In [ ]:
#@title Cell 8 — Smart Strip + Kaggle Submit
#@markdown Choose checkpoint step and strip mode
CHECKPOINT_STEP = 400  #@param {type:"integer"}
STRIP_MODE = 'smart-strip'  #@param ['smart-strip', 'v30-replica', 'no-strip']

import re as _re
from safetensors.torch import load_file, save_file

ckpt_dir = f'{CONFIG["output_dir"]}/checkpoint-{CHECKPOINT_STEP}'
if not os.path.exists(ckpt_dir):
    # Find closest checkpoint
    import glob
    ckpts = sorted(glob.glob(f'{CONFIG["output_dir"]}/checkpoint-*'),
                   key=lambda x: int(x.split('-')[-1]))
    if ckpts:
        ckpt_dir = ckpts[-1]
        print(f'Using closest checkpoint: {ckpt_dir}')
    else:
        ckpt_dir = CONFIG['output_dir']
        print(f'Using final adapter: {ckpt_dir}')

print(f'=== Smart Strip: {STRIP_MODE} ===')
sf_path = os.path.join(ckpt_dir, 'adapter_model.safetensors')
cfg_path = os.path.join(ckpt_dir, 'adapter_config.json')

tensors = load_file(sf_path)
print(f'Total keys: {len(tensors)}')

routed_re = _re.compile(r'\.experts\.\d+\.')
keep = {}
removed_routed = 0
removed_shared = 0

for key, val in tensors.items():
    if STRIP_MODE == 'v30-replica':
        if 'expert' in key.lower():
            if routed_re.search(key): removed_routed += 1
            else: removed_shared += 1
        else:
            keep[key] = val
    elif STRIP_MODE == 'smart-strip':
        if routed_re.search(key):
            removed_routed += 1
        else:
            keep[key] = val
    else:  # no-strip
        keep[key] = val

print(f'Kept: {len(keep)} | Removed routed: {removed_routed} | Removed shared: {removed_shared}')

# Save stripped
out_dir = f'/tmp/kg1_submit/stripped_{STRIP_MODE}'
os.makedirs(out_dir, exist_ok=True)
save_file(keep, os.path.join(out_dir, 'adapter_model.safetensors'))

with open(cfg_path) as f:
    cfg = json.load(f)
kept_modules = set()
for key in keep:
    for mod in ['q_proj','k_proj','v_proj','o_proj','in_proj','out_proj','up_proj','down_proj','gate']:
        if mod in key: kept_modules.add(mod)
cfg['target_modules'] = sorted(kept_modules)
with open(os.path.join(out_dir, 'adapter_config.json'), 'w') as f:
    json.dump(cfg, f, indent=2)

# Create ZIP
zip_path = '/tmp/kg1_submit/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['adapter_config.json', 'adapter_model.safetensors']:
        zf.write(os.path.join(out_dir, fname), fname)

zip_size = os.path.getsize(zip_path) / 1e6
print(f'\nZIP: {zip_path} ({zip_size:.1f} MB)')

# Verify ZIP
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    assert len(names) == 2 and all('/' not in n for n in names)
print(f'Verified: {names}')

# Submit
step_str = ckpt_dir.split('-')[-1] if 'checkpoint' in ckpt_dir else 'final'
desc = f'v51-perfect-step{step_str}-{STRIP_MODE}'
print(f'\nReady to submit: {desc}')
print(f'Run next cell to submit to Kaggle')

In [ ]:
#@title Cell 9 — Submit to Kaggle (run manually)
zip_path = '/tmp/kg1_submit/submission.zip'
step_str = ckpt_dir.split('-')[-1] if 'checkpoint' in ckpt_dir else 'final'
desc = f'v51-perfect-step{step_str}-{STRIP_MODE}'

print(f'Submitting: {desc}')
!kaggle competitions submit -c {COMPETITION} -f {zip_path} -m "{desc}"